# PM₂.₅ station forecast: reproducible audit companion

This notebook reads frozen derived outputs. It does not tune models or overwrite research results. It independently checks data quality, split integrity, model selection, held-out performance, uncertainty, high-event skill, and station transfer.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'config.json').exists(), ROOT
TABLES = ROOT / 'tables'
DERIVED = ROOT / 'data' / 'derived'
PROVENANCE = ROOT / 'provenance'
print('Experiment:', json.loads((ROOT / 'config.json').read_text())['experiment_name'])

## 1. Observation quality and immutable-source accounting

In [ ]:
obs = json.loads((PROVENANCE / 'observation_manifest.json').read_text())
quality = pd.read_csv(TABLES / 'data_quality_by_station.csv')
assert obs['source_files'] == 27 == obs['station_codes']
assert obs['conflicting_duplicate_keys'] == 0
pd.DataFrame([{
    'source_rows': obs['source_rows'],
    'unique_station_hours': obs['unique_station_hours'],
    'valid_pm25': obs['valid_pm25'],
    'valid_pm25_pct': obs['valid_pm25_pct'],
    'negative_pm25': obs['negative_pm25'],
    'pm25_at_or_above_985': obs['pm25_at_or_above_985'],
    'duplicate_rows_removed': obs['duplicate_rows_removed'],
}])

In [ ]:
display(quality[['station_code','start_utc','end_utc','valid_pm25_pct_of_expected_hours','absent_hours_within_span','invalid_temperature']].sort_values('valid_pm25_pct_of_expected_hours'))

## 2. Leakage and split audit

In [ ]:
audit = json.loads((PROVENANCE / 'modeling_table_audit.json').read_text())
assert audit['duplicate_keys'] == 0
assert audit['negative_lag_feature_definitions'] == 0
assert audit['target_before_or_at_issue'] == 0
assert audit['train_validation_target_time_overlap'] == 0
assert audit['train_test_target_time_overlap'] == 0
assert audit['validation_test_target_time_overlap'] == 0
pd.Series(audit, name='value').to_frame()

## 3. Validation-only model selection

In [ ]:
ranking = pd.read_csv(TABLES / 'model_selection_ranking.csv')
assert ranking.selected_champion.sum() == 1
champion = ranking.loc[ranking.selected_champion, 'model'].iloc[0]
print('Frozen champion:', champion)
display(ranking)

## 4. Independent 2026 test metrics

In [ ]:
metrics = pd.read_csv(TABLES / 'metrics_summary.csv')
test = metrics.query("split == 'test' and scope == 'station_balanced_common_cases'")
display(test[test.model.isin(['persistence','climatology','raw_cams','obs_lgbm','cams_lgbm','cams_xgboost','champion'])].pivot(index='forecast_hour', columns='model', values='mae_ug_m3').round(2))

In [ ]:
bootstrap = pd.read_csv(TABLES / 'block_bootstrap_skill.csv').query("model == 'champion'")
display(bootstrap[['forecast_hour','mae_improvement_ug_m3','ci95_lower_ug_m3','ci95_upper_ug_m3','n_station_weeks']].round(2))
cams_increment = pd.read_csv(TABLES / 'cams_incremental_skill_vs_observation_ml.csv', dtype={'forecast_hour': str})
display(cams_increment.query("split == 'test'")[['forecast_hour','cams_mae_improvement_over_obs_ml_ug_m3','ci95_lower_ug_m3','ci95_upper_ug_m3','bootstrap_probability_positive_pct']].round(3))

## 5. Prediction intervals and high-concentration events

In [ ]:
intervals = pd.read_csv(TABLES / 'prediction_interval_metrics.csv').query("split == 'test'")
events = pd.read_csv(TABLES / 'high_event_detection_metrics.csv')
display(intervals[['forecast_hour','n','empirical_coverage_pct','mean_interval_width_ug_m3','mean_interval_score_ug_m3']].round(2))
display(events[['forecast_hour','hits','misses','false_alarms','probability_of_detection_pct','false_alarm_ratio_pct','critical_success_index_pct']].round(2))

## 6. Spatial transfer and training-window sensitivity

In [ ]:
transfer = pd.read_csv(TABLES / 'station_transfer_metrics.csv')
transfer_summary = transfer.groupby('forecast_hour').agg(stations=('station_code','nunique'), mae_ug_m3=('mae_ug_m3','mean'), skill_vs_persistence_pct=('skill_vs_persistence_pct','mean')).round(2)
display(transfer_summary)
sensitivity = pd.read_csv(TABLES / 'recent_window_sensitivity.csv')
display(sensitivity.query("scope == 'station_balanced_common_cases' and model in ['recent_window','champion']"))

## 7. Rendered evidence

In [ ]:
display(Image(filename=str(ROOT / 'figures' / 'figure_02_test_performance.png'), width=850))
display(Image(filename=str(ROOT / 'figures' / 'figure_03_station_skill.png'), width=850))

## Interpretation boundary

Passing these checks supports retrospective predictive evaluation at the represented stations and issue cycle. It does not establish causal attribution, continuous spatial-map validity, prospective operational reliability, or equal skill at every station.